<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP34-DNS%20Tunneling%20Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import math
from collections import Counter

def shannon_entropy(s):
    if not s:
        return 0.0

    counts = Counter(s)
    length = len(s)

    return -sum(
        (count / length) * math.log2(count / length)
        for count in counts.values()
    )


def detect_dns_tunneling(queries, length_threshold=20, entropy_threshold=3.5):
    """
    Detect possible DNS tunneling by checking whether the leftmost
    domain label is both long and has high Shannon entropy.
    """
    flagged = []

    for query in queries:
        label = query.split(".")[0]
        entropy = shannon_entropy(label)

        if len(label) >= length_threshold and entropy >= entropy_threshold:
            flagged.append({
                "query": query,
                "label_length": len(label),
                "entropy": round(entropy, 2),
            })

    return flagged


# ---------------- Sample Input ----------------

queries = [
    "www.google.com",
    "mail.office365.com",
    "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com",
    "vpn.corporate-network.com",
    "9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net",
]

# ---------------- Run Detection ----------------

flagged = detect_dns_tunneling(queries)

# ---------------- Verification ----------------

flagged_domains = [item["query"] for item in flagged]

assert "www.google.com" not in flagged_domains
assert "mail.office365.com" not in flagged_domains
assert "vpn.corporate-network.com" not in flagged_domains
assert "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com" in flagged_domains
assert "9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net" in flagged_domains
assert len(flagged) == 2

print("All test cases passed.\n")

print("Potential DNS Tunneling Detected:")
for item in flagged:
    print(f"\nQuery         : {item['query']}")
    print(f"Label Length  : {item['label_length']}")
    print(f"Entropy       : {item['entropy']}")

All test cases passed.

Potential DNS Tunneling Detected:

Query         : a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com
Label Length  : 26
Entropy       : 4.47

Query         : 9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net
Label Length  : 48
Entropy       : 3.92
